In [219]:
import re, time
from pathlib import Path
from typing import Protocol
 
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")   # Kaggle không có GUI

In [220]:
LANDMARK_DIR = Path("/kaggle/input/datasets/hauuto/vietnamese-sign-language-alphabet/landmarks/landmarks/raw")
PEOPLE = ("hau", "khoi", "tai", "vy")
FNAME_RE = re.compile(r"^([a-z_]+)_([a-z]+)_([AB])_(\d+)\.npy$")
 
 
def load_all_landmarks():
    """Trả về list các dict {code, person, block, seq, arr}, arr shape (45, 63) hoặc (45, 126)."""
    records = []
    for f in sorted(LANDMARK_DIR.glob("*/*.npy")):
        m = FNAME_RE.match(f.name)
        if not m:
            print(f"CẢNH BÁO: tên file không đúng quy ước: {f.name}")
            continue
        code, person, block, seq = m.groups()
        arr = np.load(f)
        records.append({
            "code": code, "person": person, "block": block,
            "seq": int(seq), "arr": arr,
        })
    return records

def normalize_landmarks(X):
    """
    Chuẩn hóa landmark: dời về wrist + chia theo khoảng cách wrist→MCP ngón giữa.

    X: (N, T, D) với D = 63 (1 tay) hoặc 126 (2 tay)
       Mỗi tay gồm 21 điểm × 3 tọa độ (x, y, z) xen kẽ.

    Trả về: (N, T, D) float32 đã chuẩn hóa.
    """
    X_norm = X.copy().astype(np.float32)
    D = X_norm.shape[-1]
    num_coords_per_hand = 63  # 21 landmarks × 3

    # Xử lý từng bàn tay
    for hand_offset in range(0, D, num_coords_per_hand):
        hand = X_norm[..., hand_offset:hand_offset + num_coords_per_hand]
        # hand shape: (N, T, 63)

        # Tọa độ wrist (landmark 0): indices 0, 1, 2
        wrist_x = hand[..., 0:1]  # (N, T, 1)
        wrist_y = hand[..., 1:2]
        wrist_z = hand[..., 2:3]

        # Dời gốc về wrist
        hand[..., 0::3] -= wrist_x
        hand[..., 1::3] -= wrist_y
        hand[..., 2::3] -= wrist_z

        # Tọa độ middle finger MCP (landmark 9): indices 27, 28, 29
        mcp_x = hand[..., 27:28]
        mcp_y = hand[..., 28:29]
        mcp_z = hand[..., 29:30]

        # Khoảng cách wrist → MCP ngón giữa (sau khi đã dời, wrist = 0)
        dist = np.sqrt(mcp_x**2 + mcp_y**2 + mcp_z**2)
        dist = np.maximum(dist, 1e-6)  # tránh chia cho 0

        # Chia toàn bộ tọa độ cho dist → scale invariant
        hand[..., 0::3] /= dist
        hand[..., 1::3] /= dist
        hand[..., 2::3] /= dist

        X_norm[..., hand_offset:hand_offset + num_coords_per_hand] = hand

    return X_norm

def augment(
    X,
    noise_std=0.5,
    shift_range=0.05,
    zoom_range=(0.8, 1.1),
    rotation_range=(-15, 15)
):
    """
    Augmentation cho landmark bàn tay.

    X:
        (N, T, 63) hoặc (N, T, 126)

    Gồm:
        1. Gaussian Noise
        2. Shifting
        3. Zooming
        4. Spatial Rotation
    """

    X_aug = X.copy().astype(np.float32)

    # =========================
    # 1. GAUSSIAN NOISE
    # =========================
    noise = np.random.normal(
        0,
        noise_std,
        X_aug.shape
    ).astype(np.float32)

    X_aug += noise

    # =========================
    # 2. SHIFTING
    # =========================
    shift_x = np.random.uniform(
        -shift_range,
        shift_range,
        size=(X_aug.shape[0], 1, 1)
    ).astype(np.float32)

    shift_y = np.random.uniform(
        -shift_range,
        shift_range,
        size=(X_aug.shape[0], 1, 1)
    ).astype(np.float32)

    X_aug[..., 0::3] += shift_x
    X_aug[..., 1::3] += shift_y

    # =========================
    # 3. ZOOMING
    # =========================
    zoom = np.random.uniform(
        zoom_range[0],
        zoom_range[1],
        size=(X_aug.shape[0], 1, 1)
    ).astype(np.float32)

    x = X_aug[..., 0::3]
    y = X_aug[..., 1::3]

    x_center = np.mean(x, axis=-1, keepdims=True)
    y_center = np.mean(y, axis=-1, keepdims=True)

    X_aug[..., 0::3] = (
        (x - x_center) * zoom + x_center
    )

    X_aug[..., 1::3] = (
        (y - y_center) * zoom + y_center
    )

    # =========================
    # 4. SPATIAL ROTATION
    # =========================
    angles = np.random.uniform(
        rotation_range[0],
        rotation_range[1],
        size=X_aug.shape[0]
    )

    angles = np.deg2rad(angles)

    cos_a = np.cos(angles)[:, None, None]
    sin_a = np.sin(angles)[:, None, None]

    x = X_aug[..., 0::3]
    y = X_aug[..., 1::3]

    x_center = np.mean(x, axis=-1, keepdims=True)
    y_center = np.mean(y, axis=-1, keepdims=True)

    x = x - x_center
    y = y - y_center

    x_rot = x * cos_a - y * sin_a
    y_rot = x * sin_a + y * cos_a

    X_aug[..., 0::3] = x_rot + x_center
    X_aug[..., 1::3] = y_rot + y_center

    return X_aug

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, augment_fn=None):
        self.X = X          # (N, 45, D) float32
        self.y = y          # (N,) long
        self.augment_fn = augment_fn

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]     # (45, D)
        if self.augment_fn is not None:
            # augment nhận (1, 45, D), trả (1, 45, D)
            x = self.augment_fn(x[None])[0]
        return torch.tensor(x, dtype=torch.float32), self.y[idx]

class ModelStrategy(Protocol):
    def prepare_input(self, X: np.ndarray) -> np.ndarray: ...
    def train(self, X_train, y_train): ...
    def predict(self, model_state, X_test) -> np.ndarray: ...
    def measure_latency(self, model_state, X_sample) -> float: ...
 
 
def run_cross_subject(records, strategy: ModelStrategy, people=PEOPLE):
    """Train trên 3 người, test trên người còn lại, xoay vòng qua cả 4 người."""
    results = []
    for test_person in people:
        train_records = [r for r in records if r["person"] != test_person]
        test_records  = [r for r in records if r["person"] == test_person]
 
        X_train_raw = np.stack([r["arr"] for r in train_records])
        y_train     = np.array([r["code"] for r in train_records])
        X_test_raw  = np.stack([r["arr"] for r in test_records])
        y_test      = np.array([r["code"] for r in test_records])
 
        X_train = strategy.prepare_input(X_train_raw)
        X_test  = strategy.prepare_input(X_test_raw)
 
        model_state = strategy.train(X_train, y_train)
        y_pred      = strategy.predict(model_state, X_test)
        accuracy    = float(np.mean(y_pred == y_test))
        latency_ms  = strategy.measure_latency(model_state, X_test[:1])
 
        results.append({"test_person": test_person, "accuracy": accuracy, "latency_ms": latency_ms})
        print(f"Test trên {test_person}: accuracy={accuracy:.3f}, latency={latency_ms:.2f}ms")
 
    accs = [r["accuracy"] for r in results]
    lats = [r["latency_ms"] for r in results]
    print(f"\n[E2 LSTM] Trung bình: accuracy={np.mean(accs):.3f} (±{np.std(accs):.3f}), "
          f"latency={np.mean(lats):.2f}ms (±{np.std(lats):.2f}ms)")
    return results

In [221]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int,
                 num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,           # input shape: (batch, seq, feature)
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
 
    def forward(self, x):
        # x: (batch, 45, D)
        out, _ = self.lstm(x)           # out: (batch, 45, hidden_dim)
        last    = out[:, -1, :]         # lấy bước cuối: (batch, hidden_dim)
        last    = self.dropout(last)
        return self.fc(last)            # (batch, num_classes)

In [222]:
# Siêu tham số — chỉnh ở đây nếu muốn thử nghiệm
HIDDEN_DIM  = 128
NUM_LAYERS  = 3
DROPOUT     = 0.5
LR          = 1e-4
EPOCHS      = 1000
BATCH_SIZE  = 64
PATIENCE    = 10  
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
 
print(f"Device: {DEVICE}")
 
 
class E2LSTMStrategy:
    """
    E2 — LSTM một chiều trên toàn bộ chuỗi 45 bước (Vỹ).
 
    prepare_input: giữ nguyên chuỗi (N, 45, D) — KHÔNG lấy 1 khung như E0/E1.
    train        : LabelEncoder → TensorDataset → train loop.
    predict      : argmax trên logits.
    measure_latency: đo thời gian suy luận 1 mẫu, lặp 100 lần để ổn định.
    """
 
    def prepare_input(self, X: np.ndarray) -> np.ndarray:
        # E2/E3: giữ nguyên toàn bộ chuỗi (N, 45, D)
        X = X.astype(np.float32)
        # Chuẩn hóa landmark trước khi đưa vào model
        X = normalize_landmarks(X)
        return X.astype(np.float32)
 
    # ------------------------------------------------------------------
    def train(self, X_train: np.ndarray, y_train: np.ndarray):
        """
        Trả về dict chứa model, label_encoder — dùng lại ở predict/latency.
        """
        le = LabelEncoder()
        y_enc = le.fit_transform(y_train)           # string label → int
 
        num_classes = len(le.classes_)
        input_dim   = X_train.shape[2]              # 63 hoặc 126
 
        model = LSTMClassifier(
            input_dim=input_dim,
            hidden_dim=HIDDEN_DIM,
            num_layers=NUM_LAYERS,
            num_classes=num_classes,
            dropout=DROPOUT,
        ).to(DEVICE)

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_enc, test_size=0.15, stratify=y_enc, random_state=42
        )

        train_dataset = AugmentedDataset(X_tr, torch.tensor(y_tr, dtype=torch.long), augment_fn=augment)
        val_dataset   = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                       torch.tensor(y_val, dtype=torch.long))

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

        # Checkpoint: lưu model có train loss tốt nhất
        best_val_loss = float("inf")
        best_weights = None
        no_improve = 0
 
        # Thêm vào hàm train(), sau khi có y_enc
        from collections import Counter
        counts = Counter(y_tr)
        weights = torch.tensor(
            [1.0 / counts[i] for i in range(num_classes)], dtype=torch.float32
        ).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        # cosine annealing để lr giảm dần, tránh dao động cuối
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
 
        history = {"loss": [], "acc": [], "val_loss": [], "val_acc": []}
        # Lưu history của từng fold
        if not hasattr(self, "all_histories"):
            self.all_histories = []

        model.train()
        for epoch in range(1, EPOCHS + 1):
            running_loss, correct, total = 0.0, 0, 0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                logits = model(xb)
                loss   = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
 
                running_loss += loss.item() * xb.size(0)
                correct      += (logits.argmax(1) == yb).sum().item()
                total        += xb.size(0)
 
            scheduler.step()
            epoch_loss = running_loss / total
            epoch_acc  = correct / total
            history["loss"].append(epoch_loss)
            history["acc"].append(epoch_acc)

            model.eval()

            val_loss = 0.0
            val_correct = 0

            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)

                    logits = model(xb)
                    loss = criterion(logits, yb)

                    val_loss += loss.item() * xb.size(0)

                    preds = logits.argmax(dim=1)
                    val_correct += (preds == yb).sum().item()

            val_loss /= len(X_val)
            val_acc = val_correct / len(y_val)

            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            model.train()

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_weights = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                torch.save(best_weights, "E2_best_checkpoint.pt")
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"Early stopping at epoch {epoch} (val_loss={val_loss:.4f})")
                    break
 
            if epoch % 20 == 0 or epoch == 1:
                print(f"  Epoch {epoch:3d}/{EPOCHS}  loss={epoch_loss:.4f}  acc={epoch_acc:.3f}  val_loss={val_loss:.4f}")
        
        # Load lại checkpoint tốt nhất
        model.load_state_dict(best_weights)
        model.to(DEVICE)

        print(f"Loaded best checkpoint — val_loss={best_val_loss:.4f}")

        # Lưu history của fold hiện tại
        self.all_histories.append(history)
        
        return {"model": model, "le": le, "history": history, "input_dim": input_dim}
 
    # ------------------------------------------------------------------
    def predict(self, model_state, X_test: np.ndarray) -> np.ndarray:
        model = model_state["model"]
        le    = model_state["le"]
        model.eval()
        with torch.no_grad():
            X_t    = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
            logits = model(X_t)
            preds  = logits.argmax(dim=1).cpu().numpy()
        return le.inverse_transform(preds)             # trả về string label
 
    # ------------------------------------------------------------------
    def measure_latency(self, model_state, X_sample: np.ndarray) -> float:
        """Đo latency trung bình cho 1 mẫu (ms), lặp 100 lần để ổn định."""
        model = model_state["model"]
        model.eval()
        x = torch.tensor(X_sample, dtype=torch.float32).to(DEVICE)
 
        # warmup
        with torch.no_grad():
            for _ in range(10):
                model(x)
 
        # đo chính thức
        N = 100
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(N):
                model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000 / N
        return elapsed_ms

Device: cuda


In [223]:
print("=" * 60)
print("E2 — LSTM một chiều  |  cross-subject evaluation")
print("=" * 60)
 
records = load_all_landmarks()
print(f"Tổng số mẫu đọc được: {len(records)}\n")
 
strategy = E2LSTMStrategy()
strategy.all_histories = []

results = run_cross_subject(records, strategy)

E2 — LSTM một chiều  |  cross-subject evaluation
Tổng số mẫu đọc được: 640

  Epoch   1/1000  loss=3.5275  acc=0.034  val_loss=3.5282
  Epoch  20/1000  loss=3.2912  acc=0.074  val_loss=3.2685
  Epoch  40/1000  loss=2.6772  acc=0.248  val_loss=2.5771
  Epoch  60/1000  loss=2.2496  acc=0.341  val_loss=2.0450
  Epoch  80/1000  loss=1.8190  acc=0.542  val_loss=1.6860
  Epoch 100/1000  loss=1.5455  acc=0.620  val_loss=1.4558
  Epoch 120/1000  loss=1.3316  acc=0.689  val_loss=1.2094
  Epoch 140/1000  loss=1.1318  acc=0.745  val_loss=1.0427
  Epoch 160/1000  loss=0.9926  acc=0.804  val_loss=0.8925
  Epoch 180/1000  loss=0.8926  acc=0.779  val_loss=0.7890
Early stopping at epoch 195 (val_loss=0.7533)
Loaded best checkpoint — val_loss=0.7507
Test trên hau: accuracy=0.619, latency=1.07ms
  Epoch   1/1000  loss=3.5277  acc=0.027  val_loss=3.5243
  Epoch  20/1000  loss=3.2036  acc=0.081  val_loss=3.1639
  Epoch  40/1000  loss=2.6309  acc=0.218  val_loss=2.5623
  Epoch  60/1000  loss=2.2366  acc=0.

In [224]:
y_pred = strategy.predict(model_state, X_te_raw)
print("\n=== Classification Report (test_person=vy) ===")
print(classification_report(y_te, y_pred))
 
# Confusion matrix (heatmap nhỏ gọn)
classes = sorted(set(y_te) | set(y_pred))
cm = confusion_matrix(y_te, y_pred, labels=classes)
 
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=7)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — E2 LSTM (test_person=vy)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("E2_confusion_matrix.png", dpi=120)
print("Đã lưu: E2_confusion_matrix.png")
plt.close()


=== Classification Report (test_person=vy) ===
              precision    recall  f1-score   support

           a       1.00      0.50      0.67         4
          aa       0.83      0.83      0.83         6
          aw       1.00      0.83      0.91         6
           b       0.50      1.00      0.67         4
           c       0.00      0.00      0.00         4
           d       0.57      1.00      0.73         4
          dd       0.00      0.00      0.00         6
           e       1.00      0.50      0.67         4
          ee       0.00      0.00      0.00         6
           g       0.67      1.00      0.80         4
           h       0.00      0.00      0.00         4
           i       0.43      0.75      0.55         4
           k       0.43      0.75      0.55         4
           l       1.00      1.00      1.00         4
           m       0.33      1.00      0.50         4
           n       0.00      0.00      0.00         4
           o       0.44      1.00

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Đã lưu: E2_confusion_matrix.png


In [225]:
# ==================== VẼ TRAIN LOSS + VAL ACCURACY ====================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (history, test_person) in enumerate(
    zip(strategy.all_histories, PEOPLE)
):
    ax = axes[i]
    ax2 = ax.twinx()

    epochs = range(1, len(history["loss"]) + 1)

    line1, = ax.plot(
        epochs,
        history["loss"],
        label="Train loss"
    )

    line2, = ax2.plot(
        epochs,
        history["val_acc"],
        label="Val accuracy"
    )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train loss")
    ax2.set_ylabel("Val accuracy")

    ax2.set_ylim(0, 1)

    ax.set_title(
        f"E2 — Train loss & Val accuracy (test={test_person})"
    )

    ax.legend(
        handles=[line1, line2],
        loc="center right"
    )

plt.tight_layout()

plt.savefig(
    "/kaggle/working/E2_training_curve_all_folds.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("Đã lưu:")
print("/kaggle/working/E2_training_curve_all_folds.png")

Đã lưu:
/kaggle/working/E2_training_curve_all_folds.png


In [226]:
import json, pickle
 
# Lưu weights PyTorch
torch.save(model_state["model"].state_dict(), "E2_lstm_weights.pt")
 
# Lưu LabelEncoder (cần để decode prediction khi deploy)
with open("E2_label_encoder.pkl", "wb") as f:
    pickle.dump(model_state["le"], f)
 
# Lưu log kết quả cross-subject
with open("E2_cross_subject_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
 
accs = [r["accuracy"] for r in results]
lats = [r["latency_ms"] for r in results]
 
summary = {
    "experiment": "E2",
    "model": "LSTM",
    "direction": "unidirectional",
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "mean_accuracy": float(np.mean(accs)),
    "std_accuracy":  float(np.std(accs)),
    "mean_latency_ms": float(np.mean(lats)),
    "std_latency_ms":  float(np.std(lats)),
    "per_person": results,
}
 
with open("E2_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
 
print("\n=== Tóm tắt E2 ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nCác file đã lưu:")
print("  E2_lstm_weights.pt")
print("  E2_label_encoder.pkl")
print("  E2_cross_subject_results.json")
print("  E2_summary.json")
print("  E2_training_curve_all_folds.png")
print("  E2_confusion_matrix.png")


=== Tóm tắt E2 ===
{
  "experiment": "E2",
  "model": "LSTM",
  "direction": "unidirectional",
  "hidden_dim": 128,
  "num_layers": 3,
  "dropout": 0.5,
  "epochs": 1000,
  "batch_size": 64,
  "device": "cuda",
  "mean_accuracy": 0.6203125,
  "std_accuracy": 0.052361237750362645,
  "mean_latency_ms": 1.1996115774991267,
  "std_latency_ms": 0.07993515013062363,
  "per_person": [
    {
      "test_person": "hau",
      "accuracy": 0.61875,
      "latency_ms": 1.0723277400029474
    },
    {
      "test_person": "khoi",
      "accuracy": 0.58125,
      "latency_ms": 1.1907112299923028
    },
    {
      "test_person": "tai",
      "accuracy": 0.575,
      "latency_ms": 1.2692935300037789
    },
    {
      "test_person": "vy",
      "accuracy": 0.70625,
      "latency_ms": 1.2661138099974778
    }
  ]
}

Các file đã lưu:
  E2_lstm_weights.pt
  E2_label_encoder.pkl
  E2_cross_subject_results.json
  E2_summary.json
  E2_training_curve_all_folds.png
  E2_confusion_matrix.png
